# Regularization and Feature Engineering

Two critical techniques for building better ML models: **regularization** prevents overfitting by constraining model complexity, and **feature engineering** transforms raw data into better predictive signals.

## Regularization

Regularization adds a penalty term to the loss function to discourage overly complex models.

### L1 Regularization (Lasso)
$$J(\mathbf{w}) = \text{MSE} + \alpha \sum_{i=1}^{n} |w_i|$$
- Can shrink coefficients exactly to **zero** (feature selection)
- Produces sparse models

### L2 Regularization (Ridge)
$$J(\mathbf{w}) = \text{MSE} + \alpha \sum_{i=1}^{n} w_i^2$$
- Shrinks coefficients toward zero but never exactly to zero
- Handles multicollinearity well

### ElasticNet
$$J(\mathbf{w}) = \text{MSE} + \alpha_1 \sum |w_i| + \alpha_2 \sum w_i^2$$
- Combines L1 and L2 penalties
- Best of both worlds

In [ ]:
# Import libraries
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

from sklearn.datasets import load_diabetes, load_breast_cancer, load_boston
from sklearn.model_selection import train_test_split, cross_val_score
from sklearn.preprocessing import (
    StandardScaler, PolynomialFeatures, OneHotEncoder, OrdinalEncoder,
    FunctionTransformer
)
from sklearn.linear_model import LinearRegression, Ridge, Lasso, ElasticNet
from sklearn.feature_selection import SelectKBest, f_regression, RFE
from sklearn.ensemble import RandomForestRegressor
from sklearn.metrics import r2_score, mean_squared_error
from sklearn.pipeline import Pipeline

sns.set_style('whitegrid')
print("Libraries imported.")

In [ ]:
# Load Diabetes dataset
diabetes = load_diabetes()
X, y = diabetes.data, diabetes.target
feature_names = diabetes.feature_names

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42
)

# Scale
scaler = StandardScaler()
X_train_s = scaler.fit_transform(X_train)
X_test_s = scaler.transform(X_test)

print("Diabetes dataset ready.")

In [ ]:
# Compare Linear vs Ridge vs Lasso
models = {
    'Linear Regression': LinearRegression(),
    'Ridge (alpha=1.0)': Ridge(alpha=1.0, random_state=42),
    'Lasso (alpha=1.0)': Lasso(alpha=1.0, random_state=42),
    'ElasticNet (alpha=1.0)': ElasticNet(alpha=1.0, l1_ratio=0.5, random_state=42)
}

for name, model in models.items():
    model.fit(X_train_s, y_train)
    train_r2 = r2_score(y_train, model.predict(X_train_s))
    test_r2 = r2_score(y_test, model.predict(X_test_s))
    n_zero = np.sum(np.abs(model.coef_) < 1e-10) if hasattr(model, 'coef_') else 0
    print(f"  {name:30s}  Train R²={train_r2:.4f}  Test R²={test_r2:.4f}  Zero coefs={n_zero}")

In [ ]:
# Regularization path — how coefficients change with alpha
alphas = np.logspace(-3, 2, 100)
coefs_ridge = []
coefs_lasso = []

for alpha in alphas:
    ridge = Ridge(alpha=alpha).fit(X_train_s, y_train)
    lasso = Lasso(alpha=alpha, max_iter=10000).fit(X_train_s, y_train)
    coefs_ridge.append(ridge.coef_)
    coefs_lasso.append(lasso.coef_)

fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# Ridge path
for i in range(X.shape[1]):
    axes[0].semilogx(alphas, [c[i] for c in coefs_ridge], label=feature_names[i])
axes[0].set_xlabel('Alpha')
axes[0].set_ylabel('Coefficient')
axes[0].set_title('Ridge (L2) Regularization Path')
axes[0].axhline(y=0, color='gray', linestyle='--', alpha=0.5)
axes[0].legend(loc='best')

# Lasso path
for i in range(X.shape[1]):
    axes[1].semilogx(alphas, [c[i] for c in coefs_lasso], label=feature_names[i])
axes[1].set_xlabel('Alpha')
axes[1].set_ylabel('Coefficient')
axes[1].set_title('Lasso (L1) Regularization Path (coefficients go to zero)')
axes[1].axhline(y=0, color='gray', linestyle='--', alpha=0.5)
axes[1].legend(loc='best')

plt.tight_layout()
plt.show()

In [ ]:
# Finding optimal alpha with cross-validation
from sklearn.linear_model import RidgeCV, LassoCV, ElasticNetCV

ridge_cv = RidgeCV(alphas=np.logspace(-3, 2, 50), scoring='r2')
ridge_cv.fit(X_train_s, y_train)
print(f"Ridge optimal alpha: {ridge_cv.alpha_:.4f}")
print(f"Ridge CV R²: {ridge_cv.score(X_test_s, y_test):.4f}")

lasso_cv = LassoCV(alphas=np.logspace(-3, 2, 50), max_iter=10000, random_state=42)
lasso_cv.fit(X_train_s, y_train)
print(f"\nLasso optimal alpha: {lasso_cv.alpha_:.4f}")
print(f"Lasso CV R²: {lasso_cv.score(X_test_s, y_test):.4f}")
print(f"Lasso zero coefficients: {np.sum(lasso_cv.coef_ == 0)} out of {len(lasso_cv.coef_)}")

## Feature Engineering

Feature engineering is the process of creating new features from raw data to improve model performance.

### Techniques

1. **Polynomial Features**: capture non-linear relationships ($x^2$, $x^3$, $xy$)
2. **Interaction Features**: products of existing features ($x_1 \times x_2$)
3. **Binning**: convert continuous variables to categorical (age groups)
4. **Encoding**: convert categories to numbers (OneHot, Ordinal)
5. **Domain-specific features**: based on domain knowledge

In [ ]:
# Polynomial Features — capturing non-linearity
# Generate non-linear synthetic data
np.random.seed(42)
X_poly = np.linspace(-3, 3, 100).reshape(-1, 1)
y_poly = 0.5 * X_poly.ravel()**2 + X_poly.ravel() + 2 + np.random.normal(0, 1, 100)

X_p_train, X_p_test, y_p_train, y_p_test = train_test_split(X_poly, y_poly, test_size=0.2, random_state=42)

# Linear fit (underfits)
lr_poly = LinearRegression().fit(X_p_train, y_p_train)
print(f"Linear model R²: {r2_score(y_p_test, lr_poly.predict(X_p_test)):.4f}")

# Polynomial fit (degree 2)
poly = PolynomialFeatures(degree=2, include_bias=False)
X_p_train_poly = poly.fit_transform(X_p_train)
X_p_test_poly = poly.transform(X_p_test)
lr_poly2 = LinearRegression().fit(X_p_train_poly, y_p_train)
print(f"Polynomial (deg=2) R²: {r2_score(y_p_test, lr_poly2.predict(X_p_test_poly)):.4f}")

In [ ]:
# Visualize polynomial fit
X_plot = np.linspace(-3, 3, 200).reshape(-1, 1)

plt.figure(figsize=(8, 5))
plt.scatter(X_p_train, y_p_train, alpha=0.5, label='Training data')
plt.scatter(X_p_test, y_p_test, alpha=0.7, label='Test data', color='green')

# Linear
plt.plot(X_plot, lr_poly.predict(X_plot), 'r-', label='Linear (underfit)', linewidth=2)

# Polynomial
X_plot_poly = poly.transform(X_plot)
plt.plot(X_plot, lr_poly2.predict(X_plot_poly), 'b-', label='Polynomial deg=2 (good fit)', linewidth=2)

plt.xlabel('X')
plt.ylabel('y')
plt.title('Polynomial Features: Capturing Non-Linearity')
plt.legend()
plt.grid(alpha=0.3)
plt.show()

In [ ]:
# Encoding categorical variables
# Simulate a dataset with mixed feature types
np.random.seed(42)
data = {
    'age': np.random.randint(20, 70, 100),
    'income': np.random.normal(60000, 15000, 100),
    'education': np.random.choice(['High School', 'Bachelor', 'Master', 'PhD'], 100),
    'city': np.random.choice(['NYC', 'LA', 'Chicago', 'Houston'], 100),
    'target': np.random.normal(100, 20, 100)
}
df = pd.DataFrame(data)
print("Sample data with mixed types:")
print(df.head())
print(f"\nEducation categories: {df['education'].unique()}")
print(f"City categories: {df['city'].unique()}")

In [ ]:
# OneHotEncoder — creates binary columns for each category
from sklearn.preprocessing import OneHotEncoder

# Extract categorical features
cat_features = df[['education', 'city']]

ohe = OneHotEncoder(sparse_output=False, drop='first')  # drop first to avoid multicollinearity
encoded = ohe.fit_transform(cat_features)
encoded_feature_names = ohe.get_feature_names_out()

print(f"Original shape: {df.shape}")
print(f"OneHot encoded shape: {encoded.shape}")
print(f"Encoded feature names: {encoded_feature_names}")

In [ ]:
# OrdinalEncoder — preserves order for ordinal categories
from sklearn.preprocessing import OrdinalEncoder

education_order = [['High School', 'Bachelor', 'Master', 'PhD']]
oe = OrdinalEncoder(categories=education_order)
education_encoded = oe.fit_transform(df[['education']])

print("Ordinal encoding of education:")
print(pd.DataFrame({'original': df['education'].values, 'encoded': education_encoded.ravel()}).drop_duplicates().sort_values('encoded'))

In [ ]:
# Feature Selection techniques

# 1. SelectKBest — select top features based on statistical tests
selector = SelectKBest(score_func=f_regression, k=5)
X_selected = selector.fit_transform(X_train_s, y_train)
selected_indices = selector.get_support(indices=True)
print("SelectKBest (top 5 features):")
print(f"  Selected: {[feature_names[i] for i in selected_indices]}")
print(f"  Scores: {selector.scores_[selected_indices].round(2)}")

In [ ]:
# 2. RFE — Recursive Feature Elimination
rfe = RFE(estimator=LinearRegression(), n_features_to_select=5)
rfe.fit(X_train_s, y_train)
print("RFE selected features:")
print(f"  Selected: {[feature_names[i] for i, selected in enumerate(rfe.support_) if selected]}")
print(f"  Rankings: {list(zip(feature_names, rfe.ranking_))}")

In [ ]:
# 3. Feature importance from Random Forest
rf_sel = RandomForestRegressor(n_estimators=100, random_state=42)
rf_sel.fit(X_train_s, y_train)
importance = rf_sel.feature_importances_

imp_df = pd.DataFrame({'feature': feature_names, 'importance': importance})
print("Random Forest Feature Importance:")
print(imp_df.sort_values('importance', ascending=False).to_string(index=False))

In [ ]:
# Problem: Ridge/Lasso on Diabetes — find best model
print("Diabetes Regression — Model Comparison:")
print("-" * 50)

models_compare = {
    'Linear': LinearRegression(),
    'Ridge(CV)': RidgeCV(alphas=np.logspace(-3, 2, 50)),
    'Lasso(CV)': LassoCV(alphas=np.logspace(-3, 2, 50), random_state=42),
    'ElasticNet(CV)': ElasticNetCV(alphas=np.logspace(-3, 2, 50), l1_ratio=[.1, .5, .7, .9, .95, .99, 1], random_state=42)
}

for name, model in models_compare.items():
    model.fit(X_train_s, y_train)
    y_pred = model.predict(X_test_s)
    r2 = r2_score(y_test, y_pred)
    rmse = np.sqrt(mean_squared_error(y_test, y_pred))
    print(f"  {name:20s}  R²={r2:.4f}  RMSE={rmse:.2f}")

In [ ]:
# Problem: Feature engineering to improve model performance
# Add polynomial and interaction features
poly_feat = PolynomialFeatures(degree=2, interaction_only=False, include_bias=False)
X_train_poly = poly_feat.fit_transform(X_train_s)
X_test_poly = poly_feat.transform(X_test_s)

print(f"Original features: {X_train_s.shape[1]}")
print(f"With polynomial (deg=2): {X_train_poly.shape[1]}")

# Ridge on polynomial features
ridge_poly = RidgeCV(alphas=np.logspace(-3, 2, 50))
ridge_poly.fit(X_train_poly, y_train)
r2_poly = r2_score(y_test, ridge_poly.predict(X_test_poly))

print(f"\nRidge on original:      R²={r2_score(y_test, RidgeCV(alphas=np.logspace(-3, 2, 50)).fit(X_train_s, y_train).predict(X_test_s)):.4f}")
print(f"Ridge with polynomial: R²={r2_poly:.4f}")

In [ ]:
# Binning continuous variables
from sklearn.preprocessing import KBinsDiscretizer

kbd = KBinsDiscretizer(n_bins=5, encode='onehot-dense', strategy='quantile')
X_binned = kbd.fit_transform(X_train_s[:, :1])

print(f"Original single feature shape: {X_train_s[:, :1].shape}")
print(f"Binned (5 bins, one-hot) shape: {X_binned.shape}")
print(f"\nFirst 5 rows of binned feature:\n{X_binned[:5]}")

In [ ]:
# Interview Problem: Why does Lasso perform feature selection but Ridge doesn't?
print("""
Interview Question: Why does Lasso shrink coefficients to zero but Ridge doesn't?

Answer:
The key is the shape of the constraint regions:
- L1 penalty creates a DIAMOND-shaped constraint region with corners at axes
- When the elliptical contours of MSE touch the diamond at a corner,
  some coefficients become exactly zero
  
- L2 penalty creates a CIRCULAR constraint region with no sharp corners
- The optimal solution is unlikely to land exactly on an axis

Mathematically: L1's subgradient at zero includes a range, making it
possible for coefficients to be exactly zero. L2's gradient at zero is
zero, so coefficients approach but never reach zero.
""")

In [ ]:
# Summary
print("=" * 60)
print("REGULARIZATION & FEATURE ENGINEERING — KEY TAKEAWAYS")
print("=" * 60)
print("""
1. L1 (Lasso): shrinks coefficients to zero — does feature selection
2. L2 (Ridge): shrinks coefficients toward zero — handles multicollinearity
3. ElasticNet: combines L1 + L2 penalties
4. Use RidgeCV/LassoCV/ElasticNetCV to find optimal alpha
5. PolynomialFeatures captures non-linear relationships
6. OneHotEncoder for nominal categories, OrdinalEncoder for ordinal
7. Feature selection: SelectKBest, RFE, feature_importances_
8. Binning, interactions, and domain-specific features boost performance
""")